In [ ]:


from cmxflow import Workflow
from cmxflow.operators import (
    MoleculeSimilarityBlock,
    RDKitBlock,
)
from cmxflow.opt import Optimizer
from cmxflow.scores import EnrichmentScoreBlock
from cmxflow.sinks import MoleculeSinkBlock
from cmxflow.sources import MoleculeSourceBlock

#logging.basicConfig(
#    level=logging.DEBUG, format='%(asctime)s - %(levelname)s - %(message)s'
#)

In [ ]:
i = MoleculeSourceBlock()
b1 = MoleculeSimilarityBlock()
#b2 = EnumerateStereoBlock()
#b3 = make_parallel(ConformerGenerationBlock())
#b4 = MoleculeAlignBlock()
#b5 = Molecule3DSimilarityBlock()
b2 = RDKitBlock("rdkit.Chem.Descriptors.MolWt")
o = EnrichmentScoreBlock()

w = Workflow()
w.add(i, b1, b2, o)

In [ ]:
inputs = {
    '1.file@queries': 'test_crystal_ligand.sdf',
    '3.text@target': 'active',
}
w.set_required_input(inputs)
w

In [ ]:
opt = Optimizer(w, "test_benchmark.csv")
opt.optimize(n_trials=20)

In [ ]:
opt.set_best_params()
w

In [ ]:
new_o = MoleculeSinkBlock()
w.add(new_o)

In [ ]:
w('test_benchmark_small.csv', 'test-out.csv')

In [ ]:
import pandas as pd

df = pd.read_csv('test-out.csv')
df = df.sort_values('workflow_score', ascending=False)
df['frac_active'] = [
    df.iloc[:i].active.sum() / df.active.sum() for i in range(1, len(df) + 1)
]
df['frac_lib'] = [i / len(df) for i in range(1, len(df) + 1)]
ax = df.plot('frac_lib', 'frac_active')